# Sewer blockage data generation — Colab runner

Pulls the latest code from GitHub, runs the Core-4 generator, checks coverage,
and saves outputs to Google Drive. **Edit `REPO_URL` in the next cell.**

In [ ]:
REPO_URL = "https://github.com/savvy-sam/sewer-blockage.git"   # <-- EDIT THIS
import os
name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if os.path.isdir(name):
    !cd {name} && git pull --ff-only
else:
    !git clone {REPO_URL}
%cd {name}

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
# integrity check — guards against a partial/old checkout
import os, generate
print("generate.py:", os.path.getsize("generate.py"), "bytes")
print("has main():", hasattr(generate, "main"))

In [ ]:
# optional: mount Drive so results persist after the runtime ends
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# smoke test first (1 short run); confirm a parquet appears before scaling up
!python generate.py --inp inputs/BellingeSWMM_v021_nopervious.inp \
    --targets inputs/blockage_targets.csv --out data --scenarios 3 --n-per-scenario 1

In [ ]:
# full Core-4 batch (raise --n-per-scenario for more data; add --routing-step 6 for speed)
!python generate.py --inp inputs/BellingeSWMM_v021_nopervious.inp \
    --targets inputs/blockage_targets.csv --out data --scenarios 1,2,3,4 --n-per-scenario 5 --seed 0

In [ ]:
from class_summary import summarize
summarize("data")

In [ ]:
# persist outputs to Drive
dst = "/content/drive/MyDrive/sewer_datagen_outputs"
!mkdir -p "{dst}" && cp -r data/* "{dst}/" && echo "saved -> {dst}"